# [3356. Zero Array Transformation II](https://leetcode.com/problems/divide-array-into-equal-pairs/editorial/?envType=daily-question&envId=2025-03-17)

In [5]:
import math

In [75]:
'''
class SegmentTree:
    def __init__(self, nums)
    def range_add_query(self, range_l, range_r, gap)
    def range_max(self, node_i, node_l, node_r, range_l, range_r)
'''

class SegmentTree:
    def __init__(self, nums):
        self.nums_cnt = len(nums)
        self.tree = [- math.inf] * (4 * len(nums))
        self.build(1, 0, len(nums)-1, nums)

        # NOTE: This stores the updates which is to be applied to the current node and its descendants.
        self.lazy = [0] * (4 * len(nums))

    def build(self, i, node_l, node_r, nums):
        if node_l == node_r:
            self.tree[i] = nums[node_l]
            return self.tree[i]
        node_mid = (node_l + node_r) // 2
        self.tree[i] = max(
            self.build(2 * i, node_l, node_mid, nums),
            self.build(2 * i + 1, node_mid + 1, node_r, nums),
        )
        return self.tree[i]

    def range_add_query(self, range_l, range_r, gap):
        self.range_add(1, 0, self.nums_cnt - 1, range_l, range_r, gap)

    # node_i is a node which represents the segment [node_l, node_r].
    def range_add(self, node_i, node_l, node_r, range_l, range_r, gap):
        # As written in the comment of #updateRangeUtil() in https://www.geeksforgeeks.org/lazy-propagation-in-segment-tree/,
        # I need to make self.lazy[i] = 0 here when I go past this node.
        # 
        # In this way, I can do self.tree[parent_node] = max(self.tree[curr_node], self.tree[sibling_node]).
        # To acheive this, (1) I apply self.lazy[i] to self.tree[i] here (2) propagate self.lazy[i] to child nodes.
        self.tree[node_i] += self.lazy[node_i]
        if node_l + 1 <= node_r: # if not leaf
            self.lazy[2 * node_i] += self.lazy[node_i]
            self.lazy[2 * node_i + 1] += self.lazy[node_i]
        self.lazy[node_i] = 0

        # no intersection between [node_l, node_r] and [range_l, range_r]
        if range_r < node_l or node_r < range_l:
            assert self.lazy[node_i] == 0 # because already applied.
            return

        # [node_l, node_r] is included in [range_l, range_r]
        # In this case, I update this node and keep it as the lazy updates of child nodes.
        if range_l <= node_l and node_r <= range_r:
            self.tree[node_i] += gap
            if node_l + 1 <= node_r: # if not leaf
                self.lazy[2 * node_i] += gap
                self.lazy[2 * node_i + 1] += gap
            assert self.lazy[node_i] == 0
            return

        # [node_l, node_r] and [range_l, range_r] are partially overlapping.
        # Reaching here means that node_i is not a leaf node.
        assert node_l + 1 <= node_r
        
        node_mid = (node_l + node_r) // 2
        self.range_add(2 * node_i, node_l, node_mid, range_l, range_r, gap),
        self.range_add(2 * node_i + 1, node_mid + 1, node_r, range_l, range_r, gap)

        # I make sure that after self.range_add(child) is applied, the child nodes have no lazy updates.
        assert self.lazy[2 * node_i] == 0 and self.lazy[2 * node_i + 1] == 0
        self.tree[node_i] = max(
            self.tree[2 * node_i],
            self.tree[2 * node_i + 1],
        )
        assert self.lazy[node_i] == 0

    def range_max_query(self, range_l, range_r):
        return self.range_max(1, 0, len(nums)-1, range_l, range_r)

    # node_i is a node which represents the segment [node_l, node_r].
    def range_max(self, node_i, node_l, node_r, range_l, range_r):
        # Just like done in #range_add(), I propagate the lazy update of this node to children first.
        # In this way, I can simply say "range_max(this_node) = max(range_max(left_child), range_max(right_child))
        self.tree[node_i] += self.lazy[node_i]
        if node_l + 1 <= node_r: # if not leaf
            self.lazy[2 * node_i] += self.lazy[node_i]
            self.lazy[2 * node_i + 1] += self.lazy[node_i]
        self.lazy[node_i] = 0

        # [node_l, node_r] and [range_l, range_r] has no intersection.
        if node_r < range_l or range_r < node_l:
            return - math.inf

        # [node_l, node_r] is included in [range_l, range_r].
        if range_l <= node_l and node_r <= range_r:
            assert self.lazy[node_i] == 0
            return self.tree[node_i]

        # [node_l, node_r] and [range_l, range_r] are partially overlapping.
        # Reaching here means that node_i is not a leaf node.
        assert node_l + 1 <= node_r

        node_mid = (node_l + node_r) // 2
        return max(
            self.range_max(2 * node_i, node_l, node_mid, range_l, range_r),
            self.range_max(2 * node_i + 1, node_mid+1, node_r, range_l, range_r),
        )

In [76]:
nums = [1, 3, 5, 7, 9, 11]
seg_tree = SegmentTree(nums)

# Test 1: Initial max queries (before updates)
assert seg_tree.range_max_query(0, 2) == 5  # max(1,3,5) = 5
assert seg_tree.range_max_query(1, 3) == 7  # max(3,5,7) = 7
assert seg_tree.range_max_query(3, 5) == 11 # max(7,9,11) = 11

# Test 2: Adding values in a range
seg_tree.range_add_query(1, 3, 10)  # Add 10 to indices 1,2,3
assert seg_tree.range_max_query(1, 3) == 17  # max(3+10, 5+10, 7+10) = 17
assert seg_tree.range_max_query(0, 5) == 17  # Max in the whole array should be 17

# Test 3: Single element update
seg_tree.range_add_query(2, 2, 5)  # Add 5 to index 2
assert seg_tree.range_max_query(2, 2) == 20  # 5+10+5 = 20
assert seg_tree.range_max_query(1, 3) == 20  # New max in (1,3) should be 20
assert seg_tree.range_max_query(0, 5) == 20  # Whole range max should be 20

# Test 4: Full range update
seg_tree.range_add_query(0, 5, 2)  # Add 2 to all elements
assert seg_tree.range_max_query(0, 5) == 22  # Max should now be 20+2 = 22
assert seg_tree.range_max_query(3, 4) == 19  # max(7+10+2, 9+2) = 19


tree = [-inf, 11, 5, 11, 3, 5, 9, 11, 1, 3, -inf, -inf, 7, 9, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf]
